In [1]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '680eaf06c4756876c4b9140f', 'name': 'spongie01', 'fullname': 'Ali Asgar Padaria', 'email': 'aliasgarpadaria002@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': '/avatars/38e86d6f4b2650fb26861d9920bbfa93.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'kaggle-training', 'role': 'write', 'createdAt': '2026-04-10T19:01:48.165Z'}}}


/Users/aliasgar/Desktop/Ali Asgar/Projects/IMUCOCO/IMUCoCo/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!hf auth login --force


  A new version of huggingface_hub is available: 1.10.2 → 1.11.0

  Do you want to update now? [Y/n] (brew upgrade hf) ^C


In [3]:
!hf auth whoami


✓ Logged in
  user: spongie01


In [4]:
import torch
import os
from datasets import Dataset, DatasetDict

def load_pt_folder(folder_path):
    all_data = []
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith(".pt"):
            # filename format: s_{subject_id}_{seq_id}_seg{seg_id}.pt
            parts = fname.replace(".pt", "").split("_")
            subject_id = parts[1] if len(parts) > 1 else "unknown"

            data = torch.load(os.path.join(folder_path, fname), map_location="cpu")
            if isinstance(data, dict):
                row = {k: v.numpy() if hasattr(v, 'numpy') else v for k, v in data.items()}
            elif hasattr(data, 'numpy'):
                row = {"data": data.numpy()}
            else:
                row = {}

            row["subject_id"] = subject_id
            all_data.append(row)
    return Dataset.from_list(all_data)

train_split = load_pt_folder("parsed_30gigs/DIP_IMU_train_real_imu_position_only")
test_split  = load_pt_folder("parsed_30gigs/DIP_IMU_test_real_imu_position_only")

dataset = DatasetDict({
    "train": train_split,
    "test":  test_split,
})

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['joint', 'imu', 'vimu', 'gt', 'subject_id'],
        num_rows: 885
    })
    test: Dataset({
        features: ['joint', 'imu', 'vimu', 'gt', 'subject_id'],
        num_rows: 19
    })
})


In [9]:
import torch
sample = torch.load("parsed_30gigs/DIP_IMU_train_real_imu_position_only/s_01_01_seg0.pt", map_location="cpu")
print(type(sample))
print(sample.keys() if isinstance(sample, dict) else sample.shape)

<class 'dict'>
dict_keys(['joint', 'imu', 'vimu', 'gt'])


In [10]:

print(dataset["train"][0]["vimu"]["vimu_joints"])

# first 6 orientation and next 3 acceleration

[[[1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.996600329875946, -0.0606050118803978, 0.05581085383892059, 0.0592474490404129, 0.9979133009910583, 0.025667443871498108, 0.0, 0.0, 0.0], [0.9934719204902649, 0.040126778185367584, -0.10678664594888687, -0.03969884663820267, 0.9991928935050964, 0.006130944471806288, 0.0, 0.0, 0.0], [0.9995591640472412, -0.0010167474392801523, -0.029672415927052498, -0.0010167474392801523, 0.9976549744606018, -0.06843637675046921, 0.0, 0.0, 0.0], [0.9958759546279907, -0.06112949922680855, -0.06703922897577286, 0.06197236850857735, 0.9980219602584839, 0.010564076714217663, 0.0, 0.0, 0.0], [0.9985771179199219, 0.04268142953515053, -0.03196948394179344, -0.042509905993938446, 0.9990779161453247, 0.006026094313710928, 0.0, 0.0, 0.0], [0.9995689392089844, 0.0035997682716697454, 0.029137153178453445, -0.001020010095089674, 0.9961135387420654, -0.08807279169559479, 0.0, 0.0, 0.0], [0.9967880249023438, -0.05997326970100403, -0.0530744269490242, 0.05864727497100

In [11]:
dataset.push_to_hub("spongie01/DIP-IMU-position-only", private=True)


Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.61ba/s]
Processing Files (1 / 1): 100%|██████████|  313MB /  313MB,  137MB/s  
New Data Upload: 100%|██████████|  725kB /  725kB,  725kB/s  
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.74ba/s]
Processing Files (1 / 1): 100%|██████████|  311MB /  311MB,  136MB/s  
New Data Upload: 100%|██████████|  445kB /  445kB,  445kB/s  
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.74ba/s]
Processing Files (1 / 1): 100%|██████████|  311MB /  311MB,  170MB/s  
New Data Upload: 100%|██████████|  712kB /  712kB,  890kB/s  
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.79ba/s]
Processing Files (1 / 1): 100%|██████████|  311MB /  311MB,  136MB/s  
New Data Upload: 100%|██████████|  493kB /  493kB,  492kB/s  
Uploading the dataset shards: 100%|██████████| 4/4 [00:10<00:00,  2.55s/ shards]
Setting num_proc from 1 back to 1 for the test split to dis

CommitInfo(commit_url='https://huggingface.co/datasets/spongie01/DIP-IMU-position-only/commit/81f2637114c2ea5b9982673246c97298814d2c73', commit_message='Upload dataset', commit_description='', oid='81f2637114c2ea5b9982673246c97298814d2c73', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/spongie01/DIP-IMU-position-only', endpoint='https://huggingface.co', repo_type='dataset', repo_id='spongie01/DIP-IMU-position-only'), pr_revision=None, pr_num=None)